In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

# import arviz as az
from jax.random import PRNGKey

# import numpyro
# from numpyro import distributions as dist
# from numpyro import infer

pd.options.plotting.backend = "plotly"

from summer3.graph import *
from summer3.epi import *

In [ ]:
location = Stratification("location", ["North", "South"])
humans = CompartmentMap.new(location)
# Stratify into 16 5 year-age bands expressed as a 2 digit string (with the last being "75+")
age_groups = [
    "00-05",
    "05-10",
    "10-15",
    "15-20",
    "20-25",
    "25-30",
    "30-35",
    "35-40",
    "40-45",
    "45-50",
    "50-55",
    "55-60",
    "60-65",
    "65-70",
    "70-75",
    "75+",
]
age_strat = humans.stratify(Stratification("age", age_groups))
cities = humans.stratify(Stratification("city", ["A", "B"]), location["North"])

# disease_state = Stratification("disease_state", ["S", "I", "R"])
# humans = humans.stratify(disease_state)

In [ ]:
len(age_groups)

In [ ]:
times = pd.date_range("1 jan 1980", "1 jan 1981")
epi_model = CompartmentalEpiModel(humans, times)

In [ ]:
step = 7
proc_x = np.arange(0, len(times) + step, step)


proc_vals_ref = np.random.normal(
    0.0, np.linspace(0.5, 0.02, len(proc_x)), len(proc_x)
).cumsum()

cum_proc_vals = defer(jnp.cumsum)(Parameter("proc_vals", proc_vals_ref))
cum_exp_proc_vals = defer(jnp.exp)(cum_proc_vals)


def random_proc(t, cum_proc_vals):
    return jnp.interp(
        t, proc_x, cum_proc_vals, left=cum_proc_vals[0], right=cum_proc_vals[-1]
    )


def random_proc_exp(t, cum_proc_vals):
    return jnp.exp(
        jnp.interp(
            t, proc_x, cum_proc_vals, left=cum_proc_vals[0], right=cum_proc_vals[-1]
        )
    )


rp_cgf = defer(random_proc)(Time, cum_exp_proc_vals)

In [ ]:
reloc_S_to_N = TransitionFlow(
    "reloc_S_to_N",
    location["South"],
    location["North"],
    Parameter("reloc_rate", 0.1) * rp_cgf,
)

reloc_N_to_S = TransitionFlow(
    "reloc_N_to_S",
    location["North"],
    location["South"],
    Parameter("reloc_rate", 0.1),  # * rp_cgf,
)

In [ ]:
epi_model.add_flow(reloc_S_to_N)
epi_model.add_flow(reloc_N_to_S)
epi_model.set_initial_population(
    base_pops=age_strat.categories().wrap(
        np.linspace(1000, 10000, len(age_strat.strata))
    ),
    pop_splits=[location.categories().wrap(np.array([0.1, 0.9]))],
)

In [ ]:
def skew_by_age(age_skew: float = 0.0) -> CategoryData:
    rates = jnp.exp(jnp.linspace(0.0 - age_skew, age_skew, len(age_strat.strata)))
    return age_strat.categories().wrap(rates)

In [ ]:
epi_model.flows["reloc_S_to_N"].adjustments_source.append(
    defer(skew_by_age)(Parameter("age_skew_SN", 0.0))
)

epi_model.flows["reloc_N_to_S"].adjustments_dest.append(
    defer(skew_by_age)(Parameter("age_skew_NS", 0.0))
)

In [ ]:
def city_adjustment(p_a, p_b) -> CategoryData:
    return cities.categories().wrap(jnp.array([p_a, p_b]))

In [ ]:
epi_model.flows["reloc_N_to_S"].adjustments_source.append(
    defer(city_adjustment)(Parameter("city_skew_A", 0.0), Parameter("city_skew_B", 0.0))
)

In [ ]:
from summer3.runners import CompartmentalModelODE

In [ ]:
import diffrax as dfx

stepsize_controller = dfx.PIDController(rtol=1e-5, atol=1e-5, dtmax=7.0)

adjoint = dfx.RecursiveCheckpointAdjoint(checkpoints=1024)
# adjoint = dfx.DirectAdjoint()

solver_kwargs = {
    "stepsize_controller": stepsize_controller,
    "adjoint": adjoint,
    "max_steps": 10000,
}

In [ ]:
params = {
    "reloc_rate": 0.06,
    "age_skew_SN": -2.0,
    "age_skew_NS": 3.7,
    "city_skew_A": 0.12,
    "city_skew_B": 0.9,
    "proc_vals": proc_vals_ref,
}
# results = epi_model.run(params, solver_kwargs=solver_kwargs)

In [ ]:
def get_runner(epi_model, default_params, dyn_params):
    dyn_params = [
        f"parameters.{p}" for p in dyn_params if not p.startswith("parameters.")
    ]
    istate = build_istate(epi_model.cmap, epi_model.base_pops, epi_model.pop_splits)
    cmodel_ode = CompartmentalModelODE(epi_model.cmap, epi_model.flows)
    runner = cmodel_ode.get_runner(
        len(epi_model.times),
        dti_to_epoch(epi_model.times),
        computed_values=epi_model.computed_values,
        default_params=default_params,
        dyn_params=dyn_params,
    )
    return istate, runner

In [ ]:
dyn_params = [k for k in params]  # if k not in ["proc_vals"]]

In [ ]:
istate, runner = get_runner(epi_model, params, dyn_params)

In [ ]:
results = runner.run(istate.data, params, solver_kwargs=solver_kwargs)

In [ ]:
def extract_data(results):
    return results["compartments"]  # .sumcats(compartment=location.categories())

In [ ]:
ref_data = extract_data(results)

In [ ]:
def mdata(params) -> ManagedArray:
    modelled_res = runner.run(istate.data, params, solver_kwargs=solver_kwargs)
    modelled_data = extract_data(
        modelled_res
    )  # modelled_res["flows"]["reloc_N_to_S"].sumcats(source=age_strat.categories())
    return modelled_data

In [ ]:
@jit
def loss(params) -> jax.Array:
    # modelled_res = epi_model.run(params, solver_kwargs=solver_kwargs)
    # modelled_data= modelled_res["flows"]["reloc_N_to_S"].sumcats(source=age_strat.categories())
    modelled_data = mdata(params)
    return ((ref_data.data - modelled_data.data) ** 2.0).sum()

In [ ]:
gloss = jit(grad(loss))

In [ ]:
pnew = params | {
    "reloc_rate": 0.01,
    "age_skew_SN": 0.0,
    "age_skew_NS": 0.0,
    "city_skew_A": 1.0,
    "city_skew_B": 1.0,
    "proc_vals": jnp.zeros_like(proc_vals_ref),
}

In [ ]:
import optax

In [ ]:
start_params = {k: v for k, v in pnew.items() if k in dyn_params}

In [ ]:
start_learning_rate = 5e-3
optimizer = optax.adam(start_learning_rate)

# Initialize parameters of the model + optimizer.
# params = jnp.array([0.0, 0.0])
opt_state = optimizer.init(start_params)
cur_params = start_params
best_loss = loss(cur_params)
best_params = cur_params
print(best_loss)

In [ ]:
# A simple update loop.

print(loss(cur_params))

for _ in range(100):
    grads = gloss(cur_params)
    updates, opt_state = optimizer.update(grads, opt_state)
    cur_params = optax.apply_updates(cur_params, updates)
    if loss(cur_params) < loss(best_params):
        best_params = cur_params

print(loss(cur_params), loss(best_params))

In [ ]:
ref_data.to_pandas_df().plot()

In [ ]:
mdata(best_params).to_pandas_df().plot()

In [ ]:
mdata(pnew).to_pandas_df().plot()

In [ ]:
pd.DataFrame(
    {
        "ref": proc_vals_ref,
        "bestp": best_params["proc_vals"],
    }
).plot()